# Lesson 4: Prompt Evaluation

See [docs/04-prompt-evaluation.md](../docs/04-prompt-evaluation.md).

In [ ]:
import sys
from functools import partial
sys.path.insert(0, '..')

from src.init_env import set_environment_variables
from src.data_loader import load_mails, split_dataset, build_option_lists
from src.send_request import init_orchestration_service, send_request
from src.evaluation import evaluation, evalulation_full_dataset, pretty_print_table
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.models.template import Template

set_environment_variables()
init_orchestration_service()

mails = load_mails()
dev_set, test_set, test_set_small = split_dataset(mails)
categories, urgency, sentiment, option_lists = build_option_lists(mails)
mail = dev_set[10]

prompt_8 = Template(messages=[
    SystemMessage('''You are an intelligent assistant. Extract and return a json with the follwoing keys and values:
- "urgency" as one of {{?urgency}}
- "sentiment" as one of {{?sentiment}}
- "categories" list of the best matching support category tags from: {{?categories}}
Your complete message should be a valid json string that can be read directly and only contain the keys mentioned in the list above. Never enclose it in ```json...```, no newlines, no unnessacary whitespaces.

Giving the following message:'''),
    UserMessage('{{?input}}'),
])
f_8 = partial(send_request, prompt=prompt_8, **option_lists)

In [ ]:
evaluation(mail, f_8, categories)

In [ ]:
overall_result = {}
overall_result['basic--llama3.1-70b'] = evalulation_full_dataset(test_set_small, f_8, categories)
pretty_print_table(overall_result)